### 1. mask: nearest, image: linear 적용
### 2. 전처리
    Ch0: Monochrome1 inversion -> Min-Max -> 512x512 resize
    Ch1: Monochrome1 inversion -> Min-Max -> CLAHE -> 512x512 resize
    Ch2: Ch0 duplicate

In [ ]:
import os
import glob
import cv2
import pydicom
import nibabel as nib
import SimpleITK as sitk
import numpy as np
from tqdm import tqdm

TARGET_SIZE = (512, 512)

NAS_BASE = '/workspace/treat_mmtb/nas125'
DIR_IN_IMG = f"{NAS_BASE}/IDs/hyekyojeong/Data/CXR"
DIR_IN_MASK = f"{NAS_BASE}/IDs/hyekyojeong/Data/CXR_label"

DIR_OUT_BASE = f"{NAS_BASE}/IDs/hyekyojeong/model/nnUNetv2/nnUNet_data/nnUNet_raw/Dataset001_Task1"
DIR_OUT_IMG = os.path.join(DIR_OUT_BASE, "imagesTr")
DIR_OUT_LBL = os.path.join(DIR_OUT_BASE, "labelsTr")

os.makedirs(DIR_OUT_IMG, exist_ok=True)
os.makedirs(DIR_OUT_LBL, exist_ok=True)


def resize_and_pad(img, target_size=(512, 512), is_mask=False):
    h, w = img.shape[:2]
    th, tw = target_size
    scale = min(th / h, tw / w)

    nw, nh = int(w * scale), int(h * scale)

    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    resized = cv2.resize(img, (nw, nh), interpolation=interp)

    padded = np.zeros(target_size, dtype=img.dtype)
    top = (th - nh) // 2
    left = (tw - nw) // 2
    padded[top:top+nh, left:left+nw] = resized

    return padded


def save_nifti(data, save_path):
    nii = nib.Nifti1Image(data.astype(np.float32), affine=np.eye(4))
    nib.save(nii, save_path)


def save_mask_nifti(data, save_path):
    nii = nib.Nifti1Image(data.astype(np.uint8), affine=np.eye(4))
    nib.save(nii, save_path)


dcm_files = sorted(glob.glob(os.path.join(DIR_IN_IMG, "*.dcm")))
print(f"Start preprocessing: n={len(dcm_files)}")

for dcm_path in tqdm(dcm_files):
    filename = os.path.basename(dcm_path)
    our_id = os.path.splitext(filename)[0]

    mask_files = glob.glob(os.path.join(DIR_IN_MASK, f"{our_id}*.nii.gz*"))
    if not mask_files:
        print(f"Skip {our_id}: no mask")
        continue

    try:
        ds = pydicom.dcmread(dcm_path, force=True)
        raw_pixels = ds.pixel_array.astype(np.float32)

        # MONOCHROME1 inversion
        if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
            raw_pixels = np.max(raw_pixels) - raw_pixels

        # Min-Max normalization
        base_norm = (raw_pixels - raw_pixels.min()) / (raw_pixels.max() - raw_pixels.min() + 1e-8)

        # Channel 0
        ch0 = resize_and_pad(base_norm, TARGET_SIZE, is_mask=False)

        # Channel 1
        ch1_uint8 = (base_norm * 255).astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8, 8))
        ch1 = clahe.apply(ch1_uint8).astype(np.float32) / 255.0
        ch1 = resize_and_pad(ch1, TARGET_SIZE, is_mask=False)

        # Channel 2
        ch2 = ch0.copy()

        # Mask
        mask_obj = sitk.ReadImage(mask_files[0])
        raw_mask = sitk.GetArrayFromImage(mask_obj)

        if raw_mask.ndim == 3:
            raw_mask = raw_mask.squeeze()

        raw_mask = (raw_mask > 0).astype(np.uint8)
        processed_mask = resize_and_pad(raw_mask, TARGET_SIZE, is_mask=True)

        # Save
        save_nifti(ch0, os.path.join(DIR_OUT_IMG, f"{our_id}_0000.nii.gz"))
        save_nifti(ch1, os.path.join(DIR_OUT_IMG, f"{our_id}_0001.nii.gz"))
        save_nifti(ch2, os.path.join(DIR_OUT_IMG, f"{our_id}_0002.nii.gz"))
        save_mask_nifti(processed_mask, os.path.join(DIR_OUT_LBL, f"{our_id}.nii.gz"))

    except Exception as e:
        print(f"{our_id} error: {e}")

print("Done")

Start preprocessing: n=555


100%|██████████| 555/555 [02:31<00:00,  3.66it/s]

Done
